# Mission 2 Stage 14 — F01+F02 통합 입력 앙상블

각 멤버가 같은 `Clean 59D + F01 80D + F02 10D = 149D`를 입력받아 학습합니다. 세 모델은 순차 학습하고, 완료된 체크포인트·예측·지표를 검증한 뒤 고정 동일가중치 4개 앙상블을 평가합니다. Official Validation은 열지 않습니다.

**실행 순서:** 1) 경로/GPU 확인 → 2) Training 및 고정 Split 로드 → 3) F01/F02 캐시·149D 사전검사 → 4) 세 멤버 순차 학습 → 5) 앙상블 및 보고서 저장.

런타임이 끊겨 어떤 멤버가 `RUNNING` 또는 `FAILED`로 남으면 해당 디렉터리를 덮거나 자동 재학습하지 않습니다. 상태를 확인하고 새 Attempt를 별도로 설계해야 합니다.


In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
MYDRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_SHORTCUT = MYDRIVE_ROOT / 'prelim'
if not PROJECT_SHORTCUT.is_dir():
    raise FileNotFoundError(f'공유 프로젝트 바로가기에 접근할 수 없습니다: {PROJECT_SHORTCUT}')
REPO_ROOT = PROJECT_SHORTCUT.resolve(strict=True)
sys.path.insert(0, str(REPO_ROOT / 'src'))
from dcc_data import mission2_acoustic_feature_grid as grid
from dcc_data import mission2_stage14_ensemble as stage14
from dcc_data.mission2_model_family_screen import shared_training_archive_paths
from dcc_data.mission2_egemaps_colab import prepare_shared_training_raw

LAYOUT = grid.resolve_colab_drive_layout(MYDRIVE_ROOT)
SOURCE_RUN_ROOT = LAYOUT['source_run_root']
TRAINING_ARCHIVE_ROOT = LAYOUT['training_archive_root']
OUTPUT_ROOT = LAYOUT['output_root']
STAGE6_ROOT = REPO_ROOT / grid.STAGE6_REFERENCE_RELATIVE
EGEMAPS_88D = REPO_ROOT / grid.EGEMAPS_88D_RELATIVE
TRANSFER_IDENTITY = REPO_ROOT / 'tmp/mission2_egemaps_transfer_identity.json'
for path in (SOURCE_RUN_ROOT / 'feature_cache' / 'manifest.json',
             STAGE6_ROOT / 'contract.json', EGEMAPS_88D, TRANSFER_IDENTITY):
    if not path.is_file():
        raise FileNotFoundError(f'필수 Training-only 근거 파일이 없습니다: {path}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE_NAME = torch.cuda.get_device_name() if DEVICE == 'cuda' else 'CPU runtime'
print('Project:', REPO_ROOT)
print('F01/F02 source:', SOURCE_RUN_ROOT)
print('Training archives visible:', TRAINING_ARCHIVE_ROOT.is_dir())
print('Output parent:', OUTPUT_ROOT)
print('Device:', DEVICE_NAME, '| CUDA:', torch.version.cuda if DEVICE == 'cuda' else 'unavailable')


## 패키지

Feature cache를 재사용하므로 오디오 Feature 추출 패키지는 설치하지 않습니다. 이 셀을 실행하면 Colab Python 패키지가 바뀔 수 있습니다. 설치 후 import 오류가 나면 학습을 시작하지 말고 해당 환경 오류를 먼저 해결하세요.


In [ ]:
%pip -q install numpy==2.5.2 librosa==1.0.0 opensmile==2.6.0 tabm==0.0.3 rtdl-revisiting-models==0.0.2 rtdl-num-embeddings==0.0.12 pytorch-tabular==1.2.0 pyarrow pytest


## Colab Python 계약 테스트

모델 학습 전에 Stage 14 입력 결합·캐시 무결성·고정 앙상블 계산 테스트를 현재 Colab Python에서 실행합니다. 실패하면 fit 셀로 넘어가지 않습니다.


In [ ]:
import subprocess
TEST_FILE = REPO_ROOT / 'tests/test_mission2_stage14_ensemble.py'
if not TEST_FILE.is_file():
    raise FileNotFoundError(f'Stage 14 Colab test file is missing: {TEST_FILE}')
test_run = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', str(TEST_FILE)],
    text=True, capture_output=True, check=False,
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
if test_run.returncode != 0:
    raise RuntimeError('Stage 14 contract tests failed; no model fit was started')
print('COLAB_STAGE14_TESTS_PASS')


## Training 및 고정 Internal Split 불러오기

기존 임시 Training staging이 있으면 그대로 검사해 재사용하고, 없으면 Training 압축파일만 staging합니다. Official Validation은 요청하지 않습니다.


In [ ]:
from dcc_data import mission2_model_family_screen as screen

RAW_ROOT = Path('/content/mission2_stage13_training_raw')
if not RAW_ROOT.is_dir():
    RAW_ROOT = Path('/content/mission2_grid_training_raw')
if RAW_ROOT.is_dir():
    print('기존 Training staging 재사용 및 source identity 검증:', RAW_ROOT)
else:
    cache_paths = screen.models.standalone_training_cache_paths(REPO_ROOT)
    training_manifest = pd.read_parquet(
        cache_paths.manifest_parquet,
        columns=['split', 'call_id', 'audio_path', 'json_path'],
        filters=[('split', '==', 'Training')],
    )
    staging_audit = prepare_shared_training_raw(
        training_manifest,
        list(shared_training_archive_paths(REPO_ROOT / 'data/raw_original/대학부 데이터')),
        RAW_ROOT,
    )
    print(json.dumps(staging_audit, ensure_ascii=False, indent=2))
    del training_manifest

PRIOR_RUN_ROOTS = grid.stage10_to13_prior_run_roots(REPO_ROOT)
context = grid.load_training_context(
    REPO_ROOT, RAW_ROOT, prior_run_roots=PRIOR_RUN_ROOTS,
    stage6_root=STAGE6_ROOT, egemaps_cache_path=EGEMAPS_88D,
    transfer_identity_path=TRANSFER_IDENTITY,
)
split_audit = {
    'train_rows': len(context.train_idx),
    'validation_rows': len(context.validation_idx),
    'train_calls': len(np.unique(context.call_ids[context.train_idx])),
    'validation_calls': len(np.unique(context.call_ids[context.validation_idx])),
    'call_overlap': len(set(context.call_ids[context.train_idx]) & set(context.call_ids[context.validation_idx])),
    'split_seed': grid.SEED,
    'group_key': 'call_id',
    'official_validation_used': False,
}
assert split_audit['train_rows'] == 711_467, split_audit
assert split_audit['validation_rows'] == 177_891, split_audit
assert split_audit['train_calls'] == 23_360, split_audit
assert split_audit['validation_calls'] == 5_840, split_audit
assert split_audit['call_overlap'] == 0, split_audit
print(json.dumps(split_audit, ensure_ascii=False, indent=2))


## 캐시 검증 및 149D 사전검사

기존 Training 캐시의 F01/F02 파일 hash, feature 이름, 행 순서, call_id, Label provenance, finite 값을 확인합니다. 이 단계는 학습을 시작하지 않습니다.


In [ ]:
RUN_ROOT, run_audit = stage14.prepare_stage14_run(
    context, SOURCE_RUN_ROOT, OUTPUT_ROOT,
)
environment = stage14.record_environment(RUN_ROOT, device=DEVICE)
preflight = stage14.validate_preflight(context, RUN_ROOT)
assert preflight['status'] == 'PASS'
assert len(preflight['members']) == 3
print('Stage 14 Run:', RUN_ROOT)
print('Run/cache audit:', json.dumps(run_audit, ensure_ascii=False, indent=2))
print('Environment:', json.dumps(environment, ensure_ascii=False, indent=2))
print('149D preflight:', json.dumps(preflight, ensure_ascii=False, indent=2))


## 세 멤버 순차 학습

한 번에 하나씩 `TabM → Tabular ResNet → GANDALF` 순으로 실행합니다. 각 fit 완료 시 체크포인트 reload, Validation prediction, metric와 artifact hash를 확인하고 저장합니다. 중간 셀 오류가 나면 출력된 실패 상태를 먼저 검토하세요.


In [ ]:
fit_batch = stage14.run_remaining_member_fits(context, RUN_ROOT, device=DEVICE)
assert fit_batch['status'] == 'ALL_MEMBERS_COMPLETE'
print('세 통합입력 멤버 완료:', fit_batch['completed_member_count'])


## 고정 가중치 앙상블 비교 및 보고서

각 쌍은 1/2씩, 세 모델 조합은 각각 1/3씩 평균합니다. 가중치와 threshold를 탐색하지 않습니다. 결과는 accuracy, 두 speaker recall, HCE, 확률 상관, paired 교정/회귀, duration/overlap slice와 call_id bootstrap 구간을 포함합니다.


In [ ]:
result = stage14.evaluate_ensembles(context, RUN_ROOT)
print('완료:', result['status'])
print('한국어 보고서:', result['report_path'])
display(result['metrics'][['candidate_id', 'validation_accuracy', 'recall_0', 'recall_1', 'hce']])
display(result['paired_errors'])
display(result['probability_correlations'])
display(result['call_bootstrap'])
print(result['report_path'].read_text(encoding='utf-8'))
